In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Lasso
from src.encoder import preprocessor
from xgboost import XGBRegressor

In [3]:
df = pd.read_csv("../data/train_cleaned.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


We try out some feature engineering

In [20]:
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["TotalPorchSF"] = (
    df["OpenPorchSF"]
    + df["3SsnPorch"]
    + df["EnclosedPorch"]
    + df["ScreenPorch"]
    + df["WoodDeckSF"]
)

In [21]:
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)

In [22]:
df["TotalBath"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

In [23]:
df["QualTotalSF"] = df["OverallQual"] * df["TotalSF"]
df["QualGrLivArea"] = df["OverallQual"] * df["GrLivArea"]

In [24]:
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

We log transform some skewed variables

In [25]:
skewed = [
    "LotArea",
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "GarageArea",
]

for col in skewed:
    df[col] = np.log1p(df[col])

In [26]:
X_train = df.drop(columns=["SalePrice"])
Y_train = np.log1p(df["SalePrice"])

In [ ]:
model = LassoCV(cv=5, max_iter=20000, random_state=42)

In [28]:
pipe = Pipeline([
    ("pre", preprocessor(X_train)),   # IMPORTANT: uses X to detect object cols
    ("xgb", model),
])

In [29]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rmse = -cross_val_score(
    pipe,
    X_train,
    Y_train,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("CV RMSE:", rmse.mean(), "±", rmse.std())

CV RMSE: 0.12458489004786095 ± 0.01626814121000681
